In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

def sigmoid(x):
    return 1/(1+np.exp(-x))

def sigmoid_derivative(x):
    return x*(1-x)

def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1-np.square(x)

def relu(x):
    return np.maximum(0,x)

def relu_derivative(x):
    return np.where(x>0,1,0)

def activation_forward(x,func):
    if func=="sigmoid":
        return sigmoid(x)
    if func=="tanh":
        return tanh(x)
    if func=="relu":
        return relu(x)

def activation_backward(x,func):
    if func=="sigmoid":
        return sigmoid_derivative(x)
    if func=="tanh":
        return tanh_derivative(x)
    if func=="relu":
        return relu_derivative(x)

def initialize_parameters(input_size,hidden_size,output_size):
    w1=np.random.randn(input_size,hidden_size)*0.01
    b1=np.zeros((1,hidden_size))
    w2=np.random.randn(hidden_size,output_size)*0.01
    b2=np.zeros((1,output_size))
    return w1,b1,w2,b2

def forward_pass(X,w1,b1,w2,b2,activation):
    z1=np.dot(X,w1)+b1
    a1=activation_forward(z1,activation)
    z2=np.dot(a1,w2)+b2
    a2=sigmoid(z2)
    return z1,a1,z2,a2

def compute_loss(y,a2):
    m=y.shape[0]
    loss=-(1/m)*np.sum(y*np.log(a2+1e-8)+(1-y)*np.log(1-a2+1e-8))
    return loss

def backward_pass(X,y,z1,a1,a2,w2,activation):
    m=X.shape[0]
    dz2=a2-y
    dw2=(1/m)*np.dot(a1.T,dz2)
    db2=(1/m)*np.sum(dz2,axis=0,keepdims=True)
    da1=np.dot(dz2,w2.T)
    dz1=da1*activation_backward(a1,activation)
    dw1=(1/m)*np.dot(X.T,dz1)
    db1=(1/m)*np.sum(dz1,axis=0,keepdims=True)
    return dw1,db1,dw2,db2

def update_parameters(w1,b1,w2,b2,dw1,db1,dw2,db2,lr):
    w1=w1-lr*dw1
    b1=b1-lr*db1
    w2=w2-lr*dw2
    b2=b2-lr*db2
    return w1,b1,w2,b2

def train_network(X,y,hidden_size,epochs,lr,activation):
    input_size=X.shape[1]
    output_size=1
    w1,b1,w2,b2=initialize_parameters(input_size,hidden_size,output_size)
    for i in range(epochs):
        z1,a1,z2,a2=forward_pass(X,w1,b1,w2,b2,activation)
        loss=compute_loss(y,a2)
        dw1,db1,dw2,db2=backward_pass(X,y,z1,a1,a2,w2,activation)
        w1,b1,w2,b2=update_parameters(w1,b1,w2,b2,dw1,db1,dw2,db2,lr)
    return w1,b1,w2,b2

def predict(X,w1,b1,w2,b2,activation):
    _,_,_,a2=forward_pass(X,w1,b1,w2,b2,activation)
    preds=(a2>0.5).astype(int)
    return preds

def experiment(X_train,X_test,y_train,y_test):
    activations=["sigmoid","tanh","relu"]
    learning_rates=[0.01,0.05,0.1]
    hidden_units=[8,16,32]
    epochs=[500,1000]
    results=[]
    for act in activations:
        for lr in learning_rates:
            for h in hidden_units:
                for ep in epochs:
                    w1,b1,w2,b2=train_network(X_train,y_train,h,ep,lr,act)
                    preds=predict(X_test,w1,b1,w2,b2,act)
                    acc=accuracy_score(y_test,preds)
                    results.append((act,lr,h,ep,acc))
                    print("Activation:",act,"LR:",lr,"Hidden:",h,"Epochs:",ep,"Accuracy:",acc)
    return results

data=load_breast_cancer()
X=data.data
y=data.target.reshape(-1,1)

scaler=StandardScaler()
X=scaler.fit_transform(X)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

results=experiment(X_train,X_test,y_train,y_test)

best=max(results,key=lambda x:x[4])
print("Best Configuration:",best)

Activation: sigmoid LR: 0.01 Hidden: 8 Epochs: 500 Accuracy: 0.6228070175438597
Activation: sigmoid LR: 0.01 Hidden: 8 Epochs: 1000 Accuracy: 0.6842105263157895
Activation: sigmoid LR: 0.01 Hidden: 16 Epochs: 500 Accuracy: 0.6228070175438597
Activation: sigmoid LR: 0.01 Hidden: 16 Epochs: 1000 Accuracy: 0.6491228070175439
Activation: sigmoid LR: 0.01 Hidden: 32 Epochs: 500 Accuracy: 0.6228070175438597
Activation: sigmoid LR: 0.01 Hidden: 32 Epochs: 1000 Accuracy: 0.8245614035087719
Activation: sigmoid LR: 0.05 Hidden: 8 Epochs: 500 Accuracy: 0.956140350877193
Activation: sigmoid LR: 0.05 Hidden: 8 Epochs: 1000 Accuracy: 0.9824561403508771
Activation: sigmoid LR: 0.05 Hidden: 16 Epochs: 500 Accuracy: 0.9649122807017544
Activation: sigmoid LR: 0.05 Hidden: 16 Epochs: 1000 Accuracy: 0.9824561403508771
Activation: sigmoid LR: 0.05 Hidden: 32 Epochs: 500 Accuracy: 0.9649122807017544
Activation: sigmoid LR: 0.05 Hidden: 32 Epochs: 1000 Accuracy: 0.9824561403508771
Activation: sigmoid LR: 0.1